# Pipeline Flow

This notebook, `doc_reconciliationAgent.ipynb`, is an optional multi-model consolidation step after `doc_executiveSummary.ipynb` has generated per-document JSON summaries. It compares matching fields across model-output folders, then writes reconciled JSON files and audit CSV reports.

Although this notebook links to `Admission_Eligibility_Checker_using_KB.ipynb` as a next step, the current eligibility checker does **not** consume these reconciled JSON/CSV outputs. It reads combined Markdown summaries produced by `doc_executiveSummary.ipynb` instead. The reconciliation results therefore require an explicit input-code change before they can affect the eligibility decision.

# 🤖 AI Document Reconciliation Agent

**Purpose:**
This notebook acts as an automated "voting agent." It ingests document data extracted by multiple different AI models, compares their answers field-by-field, and generates a single, highly accurate "reconciled" master file.

**Output Location:**
Reconciled JSON files are saved directly inside each applicant's subdirectory under a new folder named `document summaries_reconciled`. Human-readable master reporting sheets (`reconciliation_results_long.csv`, `reconciliation_results_wide.csv`, and `reconciliation_documents_summary.csv`) are generated and saved directly in the root of your configured base Google Drive directory.

**Key Features:**
* **Cross-Model Validation:** Compares outputs from multiple LLMs (e.g., Gemma4, GLM-OCR).
* **Smart Voting Logic:** Resolves discrepancies using Consensus, Majority Vote, or First Available fallbacks.
* **Data Provenance:** Tracks exactly *why* a specific value was chosen and keeps a record of all original model answers for auditing.
* **Auto-Flattening:** Automatically handles deeply nested JSON outputs.

## 1. Setup & Environment
First, we need to import the necessary Python libraries and connect to Google Drive so the notebook can access the applicant files.
* If Drive is not mounted, a popup will ask for permission.
* We import tools for file path manipulation (`Path`), JSON handling, and data grouping (`defaultdict`).

In [ ]:
import os
from google.colab import drive
import os
import json
from pathlib import Path
from google.colab import drive
from collections import defaultdict

if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive is already mounted")

Mounting Google Drive...
Mounted at /content/drive


## 2. Configuration & Directory Validation
Here we define the ground rules for the script:
1. **`BASE_DRIVE_DIR`**: The master folder where all applicant sub-folders are stored.
2. **`MODEL_OUTPUT_DIRS`**: The exact folder names containing the JSON outputs from our different AI models. *Order matters here! The first model in this list acts as the tie-breaker.*

The code will run a quick test on the first two applicants to verify the directory structure is correct before proceeding.

In [ ]:
# 2. Define your base directory (where applicant folders are stored)
BASE_DRIVE_DIR = Path("/content/drive/MyDrive/colab-output/dream_applicant_application/")

# 3. DYNAMICALLY BUILD THE LIST OF DIRECTORIES TO RECONCILE
# We scan all applicant folders to find subdirectories starting with 'document_summaries_'
def get_dynamic_model_dirs(base_path):
    found_dirs = set()
    if not base_path.exists():
        return []

    for applicant_dir in base_path.iterdir():
        if applicant_dir.is_dir():
            for sub in applicant_dir.iterdir():
                # Identify folders that contain model output summaries
                # We skip already reconciled folders to avoid loops
                if sub.is_dir() and sub.name.startswith("document_summaries_") and "reconciled" not in sub.name:
                    found_dirs.add(sub.name)

    # Sort them to ensure consistent tie-breaking behavior (first in list is tie-breaker)
    return sorted(list(found_dirs))

MODEL_OUTPUT_DIRS = get_dynamic_model_dirs(BASE_DRIVE_DIR)

print(f"🔍 Found {len(MODEL_OUTPUT_DIRS)} model directories:")
for d in MODEL_OUTPUT_DIRS:
    print(f"  - {d}")

🔍 Found 1 model directories:
  - document_summaries___glm-ocr_granite418b


In [ ]:
import os

# Let's check the subdirectories for the first few applicants to see what actually exists
if BASE_DRIVE_DIR.exists():
    applicants = [d for d in BASE_DRIVE_DIR.iterdir() if d.is_dir()]
    if applicants:
        for applicant_dir in applicants[:2]: # Check first two applicants
            print(f"\n📁 Checking subdirectories in: {applicant_dir.name}")
            subdirs = [s.name for s in applicant_dir.iterdir() if s.is_dir()]
            if subdirs:
                for name in subdirs:
                    print(f"  - '{name}'")
            else:
                print("  (No subdirectories found)")
    else:
        print("No applicant directories found in BASE_DRIVE_DIR.")
else:
    print(f"BASE_DRIVE_DIR does not exist: {BASE_DRIVE_DIR}")


📁 Checking subdirectories in: 0_0
  - 'video_interviews'
  - 'documents_image'
  - 'documents_image_llm_text_glm-ocr'
  - 'document_summaries___glm-ocr_granite418b'

📁 Checking subdirectories in: 60811_46042
  - 'documents_image'
  - 'video_interviews'
  - 'video_interviews_audio_files'
  - 'video_interviews_audio_transcriptions_text'
  - 'video_interviews_transcriptions_grade_info_gemma-4-31b-it'
  - 'documents_image_llm_text_glm-ocr'
  - 'documents_image_text_pytesseract'
  - 'documents_image_text_pytesseract_box_info'
  - 'documents_image_llm_text_glm-ocr_doc'
  - 'document_summaries___glm-ocr_granite418b'
  - 'applicant_level_gemini_evaluations_gemini-flash-lite-latest'
  - 'document summaries_reconciled'
  - 'documents_image_llm_text_glm-ocr_doc_doc'


## 3. Document Scanning & Registry Building
Before we process anything, the script scans the entire directory tree to see what files are available.

It builds a `document_registry` dictionary that groups files logically:
`Applicant Name` ➔ `Document Name` ➔ `Model Version` ➔ `File Path`

At the end of this block, a summary is printed showing how many documents have complete data (outputs from all expected models) and are ready for full reconciliation.

In [ ]:
# 4. Data structure to hold grouped files
# Structure: document_registry[applicant_name][document_name][model_dir] = path_to_json
document_registry = defaultdict(lambda: defaultdict(dict))

print("\n🔍 Scanning directories for documents to reconcile...\n")

# 5. Scan and group the documents
if BASE_DRIVE_DIR.exists():
    for applicant_dir in BASE_DRIVE_DIR.iterdir():
        if applicant_dir.is_dir():
            applicant_name = applicant_dir.name

            # Look inside each specified model output directory
            for model_dir in MODEL_OUTPUT_DIRS:
                target_path = applicant_dir / model_dir

                if target_path.exists() and target_path.is_dir():
                    # Find all JSON files in this model's output folder
                    for json_file in target_path.glob("*.json"):
                        doc_name = json_file.name
                        # Register this file
                        document_registry[applicant_name][doc_name][model_dir] = json_file

# 6. Print a summary to verify the grouping worked
total_docs_ready = 0
for applicant, docs in document_registry.items():
    print(f"📁 Applicant: {applicant}")
    for doc_name, versions in docs.items():
        print(f"    📄 {doc_name}")
        for model_dir in versions.keys():
            print(f"       ✅ Found version from: {model_dir}")

        # Check if we have outputs from ALL expected models
        if len(versions) == len(MODEL_OUTPUT_DIRS):
            print("       ✨ Ready for Full Reconciliation (All models present)")
            total_docs_ready += 1
        else:
            print(f"       ⚠️ Partial data: {len(versions)}/{len(MODEL_OUTPUT_DIRS)} models present")
    print("-" * 40)

print(f"\n🚀 Total documents ready for full reconciliation: {total_docs_ready}")


🔍 Scanning directories for documents to reconcile...

📁 Applicant: 60811_46042
    📄 46042_CV- Saif.pdf (page 1).json
       ✅ Found version from: document_summaries___glm-ocr_granite418b
       ✨ Ready for Full Reconciliation (All models present)
    📄 46042_12th Online Result - Saif.pdf (page 1).json
       ✅ Found version from: document_summaries___glm-ocr_granite418b
       ✨ Ready for Full Reconciliation (All models present)
    📄 46042_10th certificate cum marksheet - Saif.pdf (page 1).json
       ✅ Found version from: document_summaries___glm-ocr_granite418b
       ✨ Ready for Full Reconciliation (All models present)
    📄 46042_MOI - Saif.pdf (page 1).json
       ✅ Found version from: document_summaries___glm-ocr_granite418b
       ✨ Ready for Full Reconciliation (All models present)
    📄 46042_IMG-20260812-WA0004.json
       ✅ Found version from: document_summaries___glm-ocr_granite418b
       ✨ Ready for Full Reconciliation (All models present)
-----------------------------

## 4. Core Logic & Helper Functions
This section contains the "brain" of the reconciliation agent.

**Key Functions:**
* `flatten_json`: Flattens nested JSON dictionaries into a single-level dictionary (e.g., `{"name": {"first": "John"}}` becomes `{"name.first": "John"}`). This makes field-by-field comparison much easier.
* `normalize_value`: Cleans up formatting (spaces, strings vs dicts) so the script doesn't mistake a formatting difference for a data difference.
* `choose_reconciled_value`: The voting engine.

### 🗳️ How the Voting Engine Works:
* **Null Values:** Blank/Null answers are ignored. If a model found nothing, it "abstains" from the vote.
* **Consensus:** Out of all models that found an answer, 100% of them extracted the exact same data.
* **Majority Vote:** The models disagreed, but a clear majority (e.g., 2 out of 3) agreed on a single answer.
* **First Available:** The models completely disagreed (a tie), or only a single model found data. The script defaults to the answer provided by the first model listed in `MODEL_OUTPUT_DIRS`.

In [ ]:
import pandas as pd
from collections import Counter

RECONCILED_SUBDIR = "document summaries_reconciled"

def load_json_file(file_path: Path):
    """Safely load a JSON file and return a dict (or None on failure)."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"❌ Could not read JSON: {file_path} -> {e}")
        return None

def flatten_json(data, parent_key="", sep="."):
    """Flatten nested dictionaries; keep lists as full values."""
    items = {}

    if isinstance(data, dict):
        for key, value in data.items():
            new_key = f"{parent_key}{sep}{key}" if parent_key else key
            if isinstance(value, dict):
                items.update(flatten_json(value, new_key, sep))
            else:
                items[new_key] = value
    else:
        if parent_key:
            items[parent_key] = data

    return items

def normalize_value(value):
    """Normalize values for robust cross-agent comparison."""
    if value is None:
        return None
    if isinstance(value, str):
        value = value.strip()
        return value if value else None
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False, sort_keys=True)
    return value

def to_serializable(value):
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False, sort_keys=True)
    return value

def choose_reconciled_value(values_by_model):
    """
    Resolve multiple model values using consensus, majority, or first-available.
    Note: Null/Empty values are ignored before voting begins.

    Resolution Definitions:
    - consensus: All models that returned a value provided the exact same answer.
    - majority_vote: Models disagreed, but at least 2 models agreed on the winning answer.
    - first_available: Total disagreement (tie) or only 1 model found data. Defaults to the model hierarchy.
    """
    normalized_entries = []
    for model_name in MODEL_OUTPUT_DIRS:
        if model_name in values_by_model:
            raw_val = values_by_model[model_name]
            norm_val = normalize_value(raw_val)
            if norm_val is not None:
                normalized_entries.append((model_name, raw_val, norm_val))

    if not normalized_entries:
        return None, "all_empty"

    counts = Counter(entry[2] for entry in normalized_entries)
    winner_norm, winner_count = counts.most_common(1)[0]

    if len(counts) == 1:
        method = "consensus"
    elif winner_count > 1:
        method = "majority_vote"
    else:
        method = "first_available"

    for model_name in MODEL_OUTPUT_DIRS:
        for entry_model, raw_val, norm_val in normalized_entries:
            if entry_model == model_name and norm_val == winner_norm:
                return raw_val, method

    return normalized_entries[0][1], method

## 5. The Reconciliation Execution Loop
This is where the actual work happens. For every document, the script:
1. Loads the JSON files from all available AI models.
2. Flattens the data to gather every unique field found by any model.
3. Runs the voting engine (`choose_reconciled_value`) on every single field.
4. Packages the results into a new master JSON file.
5. Saves this master file into a new `document summaries_reconciled` folder inside the applicant's directory.

**Data Provenance:** The output JSON includes a `field_provenance` section. This ensures we never lose the raw data and can trace back exactly how the script arrived at the final answer.

In [ ]:
# Reconcile JSON outputs from multiple agent/model folders
reconciliation_rows = []
document_summary_rows = []

for applicant_name, docs in document_registry.items():
    applicant_dir = BASE_DRIVE_DIR / applicant_name
    reconciled_dir = applicant_dir / RECONCILED_SUBDIR
    reconciled_dir.mkdir(parents=True, exist_ok=True)

    for doc_name, versions in docs.items():
        model_payloads = {}

        for model_dir, json_path in versions.items():
            data = load_json_file(json_path)
            if isinstance(data, dict):
                model_payloads[model_dir] = data

        if not model_payloads:
            continue

        flattened_by_model = {
            model_dir: flatten_json(payload)
            for model_dir, payload in model_payloads.items()
        }

        all_fields = sorted(set().union(*[set(d.keys()) for d in flattened_by_model.values()]))

        reconciled_fields = {}
        field_provenance = {}

        for field_name in all_fields:
            values_by_model = {
                model_dir: flattened_by_model[model_dir].get(field_name)
                for model_dir in MODEL_OUTPUT_DIRS
                if model_dir in flattened_by_model
            }

            reconciled_value, resolution_method = choose_reconciled_value(values_by_model)
            reconciled_fields[field_name] = reconciled_value

            field_provenance[field_name] = {
                "resolution_method": resolution_method,
                "values_by_model": values_by_model
            }

            row = {
                "applicant": applicant_name,
                "document": doc_name,
                "field": field_name,
                "reconciled_value": to_serializable(reconciled_value),
                "resolution_method": resolution_method
            }

            for model_dir in MODEL_OUTPUT_DIRS:
                row[f"value__{model_dir}"] = to_serializable(values_by_model.get(model_dir))

            reconciliation_rows.append(row)

        reconciled_payload = {
            "_meta": {
                "applicant": applicant_name,
                "document": doc_name,
                "source_models": list(model_payloads.keys()),
                "total_fields": len(all_fields)
            },
            "reconciled_fields": reconciled_fields,
            "field_provenance": field_provenance
        }

        reconciled_output_path = reconciled_dir / doc_name
        with open(reconciled_output_path, "w", encoding="utf-8") as f:
            json.dump(reconciled_payload, f, indent=2, ensure_ascii=False)

        document_summary_rows.append({
            "applicant": applicant_name,
            "document": doc_name,
            "source_model_count": len(model_payloads),
            "source_models": ", ".join(model_payloads.keys()),
            "reconciled_file": str(reconciled_output_path),
            "fields_reconciled": len(all_fields)
        })

print(f"✅ Reconciled {len(document_summary_rows)} documents.")
print(f"✅ Produced {len(reconciliation_rows)} field-level reconciliation records.")

✅ Reconciled 12 documents.
✅ Produced 175 field-level reconciliation records.


## 6. Analysis, Reporting & Export
Now that the JSON files are created, we convert the reconciliation data into Pandas DataFrames so it is human-readable and easy to analyze.

It generates three views:
1. **Long Format (`df_reconciliation`)**: A highly detailed, row-by-row look at every single field, showing the reconciled value, the resolution method used, and what each model originally guessed. Great for auditing `first_available` errors.
2. **Wide Format (`df_reconciled_wide`)**: A traditional database view where each row is a document and each column is a data field.
3. **Document Summary (`df_documents`)**: A high-level overview of how many documents were processed and where the output files live.

Finally, these DataFrames are exported as CSV files to the master Drive directory for downstream use.

In [ ]:
# Build DataFrames for analysis
df_reconciliation = pd.DataFrame(reconciliation_rows)
df_documents = pd.DataFrame(document_summary_rows)

if df_reconciliation.empty:
    print("No reconciliation rows found. Check directory names and JSON availability.")
else:
    print("\n📊 Field-level reconciliation DataFrame (long format):")
    #display(df_reconciliation.head(20))

    df_reconciled_wide = (
        df_reconciliation
        .pivot_table(
            index=["applicant", "document"],
            columns="field",
            values="reconciled_value",
            aggfunc="first"
        )
        .reset_index()
    )

    # Flatten pivoted columns to plain strings
    df_reconciled_wide.columns = [str(c) for c in df_reconciled_wide.columns]

    print("\n📊 Document-level reconciled DataFrame (wide format):")
    #display(df_reconciled_wide.head(10))

    print("\n📁 Reconciled document summary DataFrame:")
    #display(df_documents.head(20))

    # Optional: Persist results for downstream processing
    long_csv = BASE_DRIVE_DIR / "reconciliation_results_long.csv"
    wide_csv = BASE_DRIVE_DIR / "reconciliation_results_wide.csv"
    docs_csv = BASE_DRIVE_DIR / "reconciliation_documents_summary.csv"

    df_reconciliation.to_csv(long_csv, index=False, encoding="utf-8")
    df_reconciled_wide.to_csv(wide_csv, index=False, encoding="utf-8")
    df_documents.to_csv(docs_csv, index=False, encoding="utf-8")

    #print(f"\n💾 Saved: {long_csv}")
    #print(f"💾 Saved: {wide_csv}")
    #print(f"💾 Saved: {docs_csv}")


📊 Field-level reconciliation DataFrame (long format):

📊 Document-level reconciled DataFrame (wide format):

📁 Reconciled document summary DataFrame:


---

## ⏭️ Next Step:

[Admission_Eligibility_Checker_using_KB](https://colab.research.google.com/drive/1FXYiDOmhdmUW1mfhxW84IJsoQAiDkVxw?usp=sharing)